# 01 — EDA & Preprocessing

Explore the IMDb dataset: class distribution, review lengths, most frequent words, and the effect of cleaning.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.preprocessing import load_imdb_data, clean_text

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline


## Load Data

In [ ]:
train_df, test_df = load_imdb_data()
print(f"Train: {len(train_df):,} rows  |  Test: {len(test_df):,} rows")
train_df.head(3)


## Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, df, title in zip(axes, [train_df, test_df], ["Train", "Test"]):
    counts = df["label"].value_counts()
    ax.bar(["Negative", "Positive"], counts.values, color=["#e07b7b", "#7bc4e0"], edgecolor="white")
    ax.set_title(f"{title} Set — Class Distribution")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 80, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("../results/class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()


## Review Length Distribution

In [ ]:
train_df["raw_len"]   = train_df["text"].str.split().str.len()
train_df["clean_len"] = train_df["clean_text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title in zip(axes, ["raw_len", "clean_len"], ["Raw Text", "Cleaned Text"]):
    for label, color in [(0, "#e07b7b"), (1, "#7bc4e0")]:
        subset = train_df[train_df["label"] == label][col]
        ax.hist(subset, bins=60, alpha=0.6, color=color,
                label="Negative" if label == 0 else "Positive", edgecolor="none")
    ax.set_title(f"Review Length — {title}")
    ax.set_xlabel("Word Count")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.tight_layout()
plt.savefig("../results/length_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(train_df[["raw_len","clean_len"]].describe().round(1))


## Most Frequent Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_words(texts, n=20):
    cv = CountVectorizer(max_features=10000, stop_words="english")
    cv.fit_transform(texts)
    freq = zip(cv.get_feature_names_out(), cv.transform(texts).toarray().sum(axis=0))
    return sorted(freq, key=lambda x: x[1], reverse=True)[:n]

neg_words = top_words(train_df[train_df["label"]==0]["clean_text"])
pos_words = top_words(train_df[train_df["label"]==1]["clean_text"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, words, color, title in zip(
    axes, [neg_words, pos_words], ["#e07b7b", "#7bc4e0"], ["Negative Reviews", "Positive Reviews"]
):
    tokens, counts = zip(*words)
    ax.barh(tokens[::-1], counts[::-1], color=color, edgecolor="white")
    ax.set_title(f"Top 20 Words — {title}")
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.savefig("../results/top_words.png", dpi=120, bbox_inches="tight")
plt.show()


## Effect of Cleaning — Side by Side

In [ ]:
sample_idx = 42
raw   = train_df["text"].iloc[sample_idx]
clean = train_df["clean_text"].iloc[sample_idx]

print("ORIGINAL:")
print(raw[:400])
print("\nCLEANED:")
print(clean[:400])
